In [0]:
dbutils.widgets.text("catalog", "policybench_dev")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
# Config - adjust to match your environment
POLICYHOLDERS_TABLE = "car_insurance_claim"
TELEMATICS_TABLE = "telematics_data" 

SECRET_SCOPE = "kv-project4"
CONNECTION_STRING_SECRET = "sqldb-connection-string"

In [0]:
import re


def parse_ado_connection_string(conn_str: str) -> dict:
    """Parses a standard ADO.NET SQL connection string (Server=...;Initial Catalog=...;
    User ID=...;Password=...;) into a dict, keyed lowercase. Matches the format Azure
    Portal's SQL Database > Connection strings > ADO.NET tab gives you."""
    parts = {}
    for segment in conn_str.split(";"):
        if "=" in segment:
            key, _, value = segment.partition("=")
            parts[key.strip().lower()] = value.strip()
    return parts


raw_conn_str = dbutils.secrets.get(scope=SECRET_SCOPE, key=CONNECTION_STRING_SECRET)
conn = parse_ado_connection_string(raw_conn_str)

# "server" looks like "tcp:sql-project4.database.windows.net,1433" - strip the tcp:
# prefix and split host/port out of it
server_raw = conn.get("server", conn.get("data source", "")).replace("tcp:", "")
JDBC_HOSTNAME, _, port_str = server_raw.partition(",")
JDBC_PORT = int(port_str) if port_str else 1433
JDBC_DATABASE = conn.get("initial catalog", conn.get("database", "project4db"))
sql_user = conn.get("user id", conn.get("uid", ""))
sql_password = conn.get("password", conn.get("pwd", ""))

if not sql_password or "your_password_here" in sql_password.lower():
    raise ValueError(
        "sqldb-connection-string's password looks like the Azure Portal placeholder - "
        "update the secret in kv-project4 with the real SQL admin password before running."
    )

jdbc_url = (
    f"jdbc:sqlserver://{JDBC_HOSTNAME}:{JDBC_PORT};"
    f"database={JDBC_DATABASE};"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

connection_props = {
    "user": sql_user,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}


In [0]:
from pyspark.sql import DataFrame, functions as F
from datetime import datetime, timezone


def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
CURRENCY_COLS = ["INCOME", "HOME_VAL", "BLUEBOOK", "OLDCLAIM", "CLM_AMT"]


def clean_policyholders(df: DataFrame) -> DataFrame:
    # currency columns come in as "$67,349" strings - strip $ and , then cast
    for c in CURRENCY_COLS:
        df = df.withColumn(c, F.regexp_replace(F.col(c), r"[\$,]", "").cast("double"))

    # ID is NOT a unique key - same pattern already confirmed on the Fabric build:
    # one person can hold multiple insured vehicles/policies, each a separate row
    # with different vehicle/claim data. Do NOT dedupe by ID - that drops real
    # records. Surrogate POLICY_ID is the true row grain; ID stays as the person link.
    df = df.withColumn("POLICY_ID", F.monotonically_increasing_id())
    return df


def parse_telematics_events(df: DataFrame) -> DataFrame:
    # timeMili (ms epoch) is null on a large share of rows across nearly every PID -
    # the human-readable "timestamp" string col has a real value whenever timeMili
    # doesn't (confirmed on the Fabric build against raw source data). Fall back to
    # parsing that string instead of dropping it. Second-precision only (coarser
    # than a genuine timeMili reading), but still real per-reading time.
    parsed_ts = F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd HH:mm:ss.SSSSSS")
    fallback_ms = F.unix_timestamp(parsed_ts) * 1000
    df = df.withColumn(
        "timeMili",
        F.when(F.col("timeMili").isNotNull(), F.col("timeMili")).otherwise(fallback_ms)
    )
    df = df.drop("timestamp")

    # rename to the canonical event schema used everywhere downstream (matches Fabric)
    df = (
        df.withColumnRenamed("deviceId", "device_id")
        .withColumnRenamed("timeMili", "timestamp")
        .withColumnRenamed("variable", "PID")
        .withColumnRenamed("alarmClass", "alarm_class")
    )

    # bad/unlabeled PID - drop, not worth cleaning
    df = df.filter(F.col("PID") != "PIDNameNotAvailable")

    # POSITION's value is "lat,lon,alt" - split into 3 scalar PID rows so every
    # row in the output has one numeric value, same as every other PID
    position = df.filter(F.col("PID") == "POSITION")
    parts = F.split(F.col("value"), ",")
    position_lat = position.withColumn("PID", F.lit("POSITION_LAT")).withColumn(
        "value", parts.getItem(0).cast("double")
    )
    position_lon = position.withColumn("PID", F.lit("POSITION_LON")).withColumn(
        "value", parts.getItem(1).cast("double")
    )
    position_alt = position.withColumn("PID", F.lit("POSITION_ALT")).withColumn(
        "value", parts.getItem(2).cast("double")
    )

    scalar = df.filter(F.col("PID") != "POSITION").withColumn("value", F.col("value").cast("double"))

    events = scalar.unionByName(position_lat).unionByName(position_lon).unionByName(position_alt)
    events = events.withColumn("timestamp", F.col("timestamp").cast("long"))
    events = events.select("device_id", "timestamp", "PID", "value", "alarm_class")

    # null handling - drop rows where the cast failed (bad source value)
    events = events.dropna(subset=["value"])
    return events


In [0]:


start_dt = datetime.now(timezone.utc)

raw_policyholders = spark.read.jdbc(url=jdbc_url, table=POLICYHOLDERS_TABLE, properties=connection_props)
raw_telematics = spark.read.jdbc(url=jdbc_url, table=TELEMATICS_TABLE, properties=connection_props)

In [0]:

bronze_policyholders = clean_policyholders(raw_policyholders)
bronze_policyholders.write.format("delta").mode("overwrite").saveAsTable("bronze_car_insurance_claim")

bronze_telematics = parse_telematics_events(raw_telematics)
bronze_telematics.write.format("delta").mode("overwrite").saveAsTable("bronze_telematics_events")

end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "bronze", start_dt, end_dt)


In [0]:
bronze_policyholders_check = spark.read.table("bronze_car_insurance_claim")
bronze_telematics_check = spark.read.table("bronze_telematics_events")

print("bronze_car_insurance_claim rows:", bronze_policyholders_check.count())
print("bronze_telematics_events rows:", bronze_telematics_check.count())
print("bronze_telematics_events rows with null timestamp:", bronze_telematics_check.filter(F.col("timestamp").isNull()).count())